# F1 medallion pipeline (Microsoft Fabric lakehouse)

Raw CSVs from the Kaggle *Formula 1 World Championship* dataset sit in `Files/`.

- **Bronze**: the raw CSV files in `Files/`
- **Silver**: typed Delta tables in the `silver` schema (the dataset marks missing values as `\N`, converted to NULL)
- **Gold**: a star schema in the `gold` schema (dimensions + facts), including sprint-race points

Run the cells top to bottom. The notebook needs `f1_lakehouse` set as the default lakehouse.

## Bronze to silver

In [ ]:
from pyspark.sql import functions as F

TABLES = [
    "circuits", "constructor_results", "constructor_standings", "constructors",
    "driver_standings", "drivers", "lap_times", "pit_stops", "qualifying",
    "races", "results", "seasons", "sprint_results", "status",
]

def load_csv(name):
    return (spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("nullValue", "\\N")   # missing values are the literal text \N
            .csv(f"Files/{name}.csv"))

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

for name in TABLES:
    df = load_csv(name)
    df.write.mode("overwrite").format("delta").saveAsTable(f"silver.{name}")
    print(f"{name}: {df.count():,} rows")

## Silver to gold (star schema)

In [ ]:
assert dict(spark.table("silver.results").dtypes)["position"] == "int", \
    "position is not int, check nullValue in load_csv"

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

def silver(name):
    return spark.table(f"silver.{name}")

def save(df, name):
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(f"gold.{name}"))
    print(f"gold.{name}: {df.count():,} rows")

### Dimensions

In [ ]:
save(silver("drivers").select(
    "driverId", "code", "nationality", "dob",
    F.concat_ws(" ", "forename", "surname").alias("driver_name")),
    "dim_driver")

save(silver("constructors").select(
    "constructorId", F.col("name").alias("constructor_name"), "nationality"),
    "dim_constructor")

save(silver("status").select("statusId", "status"), "dim_status")

# circuit attributes are folded into the race dimension to keep a pure star
circuits = silver("circuits").select(
    "circuitId", F.col("name").alias("circuit_name"),
    F.col("location").alias("circuit_location"),
    F.col("country").alias("circuit_country"), "lat", "lng")

save(silver("races").select(
        "raceId", "year", "round", "circuitId",
        F.col("name").alias("race_name"), F.col("date").alias("race_date"))
    .join(circuits, "circuitId", "left")
    .drop("circuitId"),
    "dim_race")

### Facts

In [ ]:
# race results, with sprint-race points added (sprints exist since 2021)
sprint = (silver("sprint_results")
          .select("raceId", "driverId", F.col("points").alias("sprint_points"))
          .dropDuplicates(["raceId", "driverId"]))

save(silver("results").select(
        "resultId", "raceId", "driverId", "constructorId", "statusId",
        "grid", "position", "positionOrder", "points", "laps", "milliseconds",
        "fastestLap", F.col("rank").alias("fastest_lap_rank"), "fastestLapSpeed")
    .join(sprint, ["raceId", "driverId"], "left")
    .withColumn("sprint_points", F.coalesce(F.col("sprint_points"), F.lit(0.0)))
    .withColumn("total_points", F.col("points") + F.col("sprint_points"))
    .withColumn("is_win", (F.col("positionOrder") == 1).cast("int"))
    .withColumn("is_podium", (F.col("positionOrder") <= 3).cast("int"))
    .withColumn("is_dnf", F.col("position").isNull().cast("int")),
    "fact_results")

In [ ]:
# laps and pit stops get the constructor via the results table
bridge = (silver("results")
          .select("raceId", "driverId", "constructorId")
          .dropDuplicates(["raceId", "driverId"]))

save(silver("lap_times").select(
        "raceId", "driverId", "lap",
        F.col("position").alias("lap_position"), "milliseconds")
     .join(bridge, ["raceId", "driverId"], "left"),
     "fact_laps")

save(silver("pit_stops").select(
        "raceId", "driverId", "stop", "lap",
        F.col("milliseconds").alias("pit_ms"))
     .join(bridge, ["raceId", "driverId"], "left"),
     "fact_pit_stops")

## Validation

Expected for 2024: Verstappen 437, Norris 374, Leclerc 356.

In [ ]:
print(spark.table("gold.fact_results").columns)

spark.sql("""
    SELECT d.driver_name, SUM(f.total_points) AS points
    FROM gold.fact_results f
    JOIN gold.dim_driver d ON f.driverId = d.driverId
    JOIN gold.dim_race r ON f.raceId = r.raceId
    WHERE r.year = 2024
    GROUP BY d.driver_name
    ORDER BY points DESC
    LIMIT 5
""").show()